# Lab 2: MinIO với Docker

## 🎯 Mục tiêu
- Chạy **MinIO** bằng Docker Compose
- Truy cập **MinIO Console** (Web UI) để tạo bucket, upload/download object
- Kiểm tra kết nối từ Python (chuẩn bị cho Lab 3)

## 📋 Prerequisites
- Docker và Docker Compose đã cài
- Chạy từ thư mục `S3_lab`: `docker compose up -d`


## 1. Khởi động MinIO (chạy trong terminal)

```bash
cd S3_lab
docker compose up -d
docker compose ps
```

- **Port 9000**: S3 API (client sẽ kết nối tới đây)
- **Port 9001**: MinIO Console (Web UI)
- Credentials mặc định: `minioadmin` / `minioadmin`


## 2. Truy cập MinIO Console

1. Mở trình duyệt: **http://localhost:9001**
2. Đăng nhập: `minioadmin` / `minioadmin`
3. **Create Bucket**: tạo bucket (vd: `my-bucket`)
4. **Upload**: vào bucket → Upload → chọn file
5. **Download**: click object → Download

Thử tạo bucket `lab-bucket` và upload một file text hoặc CSV để làm quen.

## 3. Kiểm tra MinIO từ Python

Cell bên dưới dùng **boto3** (S3 client) kết nối tới MinIO. Cần:
- MinIO đang chạy (`docker compose up -d`)
- Đã cài: `pip install boto3` (hoặc chạy `./setup_s3_lab.sh`)

In [ ]:
import boto3
from botocore.client import Config

ENDPOINT = "http://localhost:9000"
ACCESS_KEY = "minioadmin"
SECRET_KEY = "minioadmin"
REGION = "us-east-1"

s3 = boto3.client(
    "s3",
    endpoint_url=ENDPOINT,
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION,
    config=Config(signature_version="s3v4"),
)

response = s3.list_buckets()
buckets = [b["Name"] for b in response.get("Buckets", [])]
print("Buckets hiện có:", buckets if buckets else "(chưa có bucket)")
print("✅ Kết nối MinIO thành công.")

## 4. Tạo bucket từ code (tùy chọn)

Nếu chưa tạo bucket qua Console, có thể tạo bằng code:

In [ ]:
BUCKET = "lab-bucket"
try:
    s3.create_bucket(Bucket=BUCKET)
    print(f"✅ Đã tạo bucket: {BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket {BUCKET} đã tồn tại.")
except Exception as e:
    print(f"Lỗi: {e}")

## 5. Dừng MinIO (khi xong lab)

```bash
docker compose down
```

Dữ liệu trong volume `minio_data` vẫn giữ. Để xóa cả volume: `docker compose down -v`.

---

**Tiếp theo**: Lab 3 – S3 API & SDK (upload/download/list bằng boto3).